In [85]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
import sklearn
import scipy 
import linearmodels.panel as lmp
import seaborn as sns
from linearmodels.iv import IV2SLS

import warnings
warnings.filterwarnings("ignore")

%matplotlib inline

In [86]:
eps=pd.read_csv('../data/panel_eps.csv')
eps = eps.drop(columns=['children', 'publicemp','size','lmp', 'fondoa', 'fondob', 'fondoc', 'fondod', 'fondoe']) #se dropean todas las columnas que no tienen relevancia para el estudio
vars=eps.columns.values.tolist()
print(vars)
# bycount = eps['year'].groupby(eps['year']).count()
# bIDcount = eps['ID'].groupby(eps['ID']).count()
# eps['yr'] = eps['year'].astype(object)

# eps = eps.set_index(["ID","year"])
# eps.describe()

['folio_n20', 'year', 'time', 'edad', 'hombre', 'edu', 'region', 'status', 'kids', 'situation', 'exp', 'occupation', 'wage', 'hours', 'informal', 'selfemp', 'sistema', 'cotizando', 'assets', 'exp_sist', 'expectancy', 'illness', 'cronica', 'nocronica', 'mental']


In [87]:
# Reemplaza todos los números mayores a 200 por 2025 menos el número
eps = eps.applymap(lambda x: 2025 - x if isinstance(x, (int, float)) and x > 200 else x)

# Reemplaza los NaN en las columnas de salud por 0
eps['illness'] = eps['illness'].fillna(0)
eps['cronica'] = eps['cronica'].fillna(0)
eps['nocronica'] = eps['nocronica'].fillna(0)
eps['mental'] = eps['mental'].fillna(0)
# Reemplaza los valores de 'expectancy' mayores a 300 por NaN
eps.loc[eps['expectancy'] > 300, 'expectancy'] = np.nan

eps = pd.get_dummies(eps, columns=['situation', 'status'], dtype='int')
eps['wage'] = eps['wage'].dropna()
eps['wage'] = eps['wage'] * -1
eps.loc[(eps['wage'].isna()) & ((eps['situation_2'] == 1) | (eps['situation_4'] == 1)), 'wage'] = 0
eps['wage'] = eps['wage'].dropna()
eps['edu'] = eps['edu'].dropna()
eps['folio_n20'] = eps['folio_n20']*-1
eps.loc[eps['exp_sist'] >0, 'exp_sist'] = 1
eps.describe()


,folio_n20,year,time,edad,hombre,edu,region,kids,exp,occupation,...,mental,situation_1,situation_2,situation_3,situation_4,status_1.0,status_2.0,status_3.0,status_4.0,status_5.0
count,9.684600e+04,96846.000000,96846.000000,96846.000000,96796.000000,92920.000000,80848.000000,96846.000000,84869.000000,49777.000000,...,96846.000000,96846.000000,96846.000000,96846.000000,96846.000000,96846.000000,96846.000000,96846.000000,96846.000000,96846.000000
mean,1.253178e+11,7.991213,3.493898,41.191872,0.494979,7.905585,9.148959,0.357082,12.749143,5.836692,...,0.061778,0.002581,0.083700,0.576059,0.337660,0.454195,0.107614,0.085032,0.063833,0.288189
std,1.296642e+09,4.566324,1.727313,12.649125,0.499977,5.106440,3.678007,0.479142,9.563255,2.463750,...,0.240754,0.050742,0.276939,0.494184,0.472914,0.497900,0.309894,0.278931,0.244457,0.452922
min,1.245602e+11,2.000000,1.000000,20.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.246212e+11,4.000000,2.000000,33.000000,0.000000,3.000000,6.000000,0.000000,4.000000,4.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.247076e+11,6.000000,3.000000,39.000000,0.000000,7.000000,9.000000,0.000000,12.000000,6.000000,...,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.248369e+11,12.000000,5.000000,43.000000,1.000000,12.000000,13.000000,1.000000,21.000000,8.000000,...,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000
max,1.281337e+11,15.000000,6.000000,111.000000,1.000000,19.000000,15.000000,1.000000,36.000000,10.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [ ]:

y=eps['folio_n20']
X=eps['edu', 'wage', 'exp_sist', 'cotizando']
X=sm.add_constant(X)
model = lmp.PanelOLS(y, X)
mco = model.fit()
print(mco)

KeyError: ('edu', 'wage', 'exp_sist', 'cotizando')